# 3. Model comparison and RNA contributions

**Question:** How well do the models work, and which assay targets influence their scores?

This notebook walks through the completed binary comparison using its saved split,
models, and thresholds. It recomputes scores and checks them against the saved
predictions. New feature-contribution plots are retrospective descriptions added
September 27, 2026, after the presentation.

The four comparisons are a training-prevalence constant, IFNG alone, regularized
multivariable logistic regression, and the selected CatBoost model. All evaluation
rows are the same 345 technical-validation specimens. The original experiment also
screened other candidates; their discovery results are shown for transparency.

## Setup

Run with this project's Python 3.12 environment (uv sync --frozen --group notebooks).
The [notebook guide](README.md) has launch and clean-checkout instructions.
Each notebook runs independently. Figures and aggregate tables go to a new
directory under results/notebooks/; complete specimen tables stay in memory.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src/kidney_biopsy").is_dir():
    raise RuntimeError("Start Jupyter from this project root or its notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from kidney_biopsy import map_diagnoses, normalize_counts, read_geo_matrix, read_rcc_archive
from notebooks.notebook_support import (
    BK_TARGETS,
    COLORS,
    DIAGNOSIS_ORDER,
    SHORT_DIAGNOSES,
    NotebookSession,
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 4.5),
        "figure.dpi": 110,
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "axes.titlepad": 14,
    }
)
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 3)
session = NotebookSession("03_model_comparison_and_rna_contributions")
display(session.verify_inputs())

In [ ]:
raw_dir = ROOT / "data/raw/rejection_public"
deposited_expression, metadata = read_geo_matrix(raw_dir / "GSE212160_series_matrix.txt.gz")
counts, batches = read_rcc_archive(raw_dir / "GSE212160_RAW.tar", specimen_ids=metadata.index)
diagnosis = metadata.histology_diagnosis
y = map_diagnoses(diagnosis)

In [ ]:
import joblib
from catboost import Pool
from IPython import get_ipython
from scipy.special import expit
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve

from experiments.rejection_public.run import choose_threshold
from kidney_biopsy import load_predictor, predict_scores
from scripts.analyze_results import align_predictions, metrics, paired_bootstrap, reliability

# The standalone report helper uses Agg; restore inline figures for this notebook.
get_ipython().run_line_magic("matplotlib", "inline")

split = session.load_split(metadata)
train_ids = split.index[split.split.eq("train")]
screen_ids = split.index[split.split.eq("discovery_screen")]
validation_ids = split.index[split.split.eq("author_validation")]
frozen = json.loads(session.artifact("any_rejection_frozen.json").read_text())
configuration = json.loads(session.artifact("configuration.json").read_text())
saved_results = json.loads(session.artifact("biopsy_results.json").read_text())["any_rejection"]
predictor = load_predictor(ROOT, session.run_dir)
session.artifact("any_rejection_selected_model.joblib", model=True)
X = normalize_counts(counts, predictor.schema)
selected = frozen["selected_model"]
assert selected == "catboost_all_depth4", (
    "This walkthrough explains the preserved depth-4 CatBoost run."
)
print(f"Run: {session.run_dir}")
print(f"Model version: {predictor.model_version}")
print(
    f"Training / screening / technical validation: {len(train_ids)} / {len(screen_ids)} / {len(validation_ids)}"
)

## 1. How were the models fitted and selected?

The [research script](../experiments/rejection_public/run.py) creates and fits the
candidates. All use the same within-specimen housekeeping normalization.
Logistic regression standardizes targets using training means and standard
deviations. Selected-feature candidates also select their targets on training only.
CatBoost uses the normalized targets directly.

This notebook loads trusted, hash-verified models from the completed project run.
The [guide](README.md#reproduce-from-a-clean-checkout) shows how to reproduce
training in new directories. Re-executing this notebook does not silently retrain.

In [ ]:
recipe = pd.DataFrame(
    [
        ("Constant", "Training rejection fraction", "No expression inputs"),
        ("IFNG", "StandardScaler + logistic regression, C=1", "One assay target"),
        ("Logistic", "StandardScaler + logistic regression, C=0.1", "758 assay targets"),
        ("CatBoost", "300 trees, depth 4, learning rate 0.04, seed 2026", "758 assay targets"),
    ],
    columns=["comparison", "training_recipe", "inputs"],
)
display(recipe)
screening = pd.read_csv(session.artifact("biopsy_screen.csv"))
screening = screening.loc[screening.target.eq("any_rejection")].set_index("model")
display(
    screening[
        ["features", "threshold", "tp", "fn", "fp", "tn", "sensitivity", "specificity", "roc_auc"]
    ]
)
session.table("discovery_screening", screening)

## 2. Reconstruct threshold selection on screening specimens

For each expression model, choose the highest screening threshold retaining at
least 90% of recorded rejection cases. Among multivariable candidates, maximize
screening specificity, then use ROC-AUC to break ties. IFNG is a separate benchmark.
The fitted models are retained without refitting. The constant uses threshold 0.5.

This 90% rule was an experiment choice. It does not guarantee 90% sensitivity in
another cohort, and it does not calibrate scores into reliable probabilities.

In [ ]:
names = [selected, "logistic_all", "single_gene_IFNG"]
labels = {
    selected: "CatBoost",
    "logistic_all": "Logistic",
    "single_gene_IFNG": "IFNG",
    "training_prevalence": "Constant",
}
models, thresholds, scores, checks = {}, {}, {}, []
for name in names:
    models[name] = joblib.load(session.artifact(f"any_rejection_{name}.joblib", model=True))
    features = X[["IFNG"]] if name == "single_gene_IFNG" else X
    thresholds[name] = float(screening.loc[name, "threshold"])
    saved_screen = pd.read_csv(
        session.artifact(f"any_rejection_{name}_screen_predictions.csv"), index_col="sample"
    ).rename(columns={"rejection_score": "probability"})
    saved_screen = align_predictions(saved_screen, split.loc[screen_ids], thresholds[name])
    actual_screen = predict_scores(models[name], features.loc[screen_ids])
    np.testing.assert_allclose(actual_screen, saved_screen.probability, rtol=0, atol=1e-10)
    reconstructed = choose_threshold(saved_screen.y.to_numpy(), saved_screen.probability.to_numpy())
    np.testing.assert_allclose(reconstructed, thresholds[name], rtol=0, atol=1e-12)
    scores[name] = predict_scores(models[name], features.loc[validation_ids])
    saved_test = pd.read_csv(
        session.artifact(f"any_rejection_{name}_test_predictions.csv"), index_col="sample"
    )
    saved_test = align_predictions(saved_test, split.loc[validation_ids], thresholds[name])
    np.testing.assert_allclose(scores[name], saved_test.probability, rtol=0, atol=1e-10)
    np.testing.assert_array_equal(scores[name] >= thresholds[name], saved_test.predicted)
    checks.append(
        {
            "model": name,
            "max_score_difference": float(np.max(np.abs(scores[name] - saved_test.probability))),
            "screen_threshold_reproduced": True,
            "validation_flags_match": True,
        }
    )
thresholds["training_prevalence"] = 0.5
scores["training_prevalence"] = np.full(len(validation_ids), y.loc[train_ids].mean())
display(pd.DataFrame(checks))
eligible = screening.drop(index="single_gene_IFNG")
winner = eligible.sort_values(["specificity", "roc_auc"], ascending=False).index[0]
assert winner == selected
session.table("score_and_threshold_checks", pd.DataFrame(checks))

## 3. Compare missed rejection and false flags on the same rows

A missed rejection is a recorded rejection below the model's fixed threshold.
A false flag is a no-rejection specimen at or above it. Confusion matrices use
recorded diagnoses on the rows and model flags on the columns.

In [ ]:
y_validation = y.loc[validation_ids].to_numpy()
metric_table = pd.DataFrame(
    {name: metrics(y_validation, score, thresholds[name]) for name, score in scores.items()}
).T
for name in scores:
    for field in ["tp", "fn", "fp", "tn"]:
        assert int(metric_table.loc[name, field]) == saved_results["author_validation"][name][field]
display(
    metric_table[
        [
            "threshold",
            "fn",
            "fp",
            "sensitivity",
            "specificity",
            "ppv",
            "npv",
            "roc_auc",
            "average_precision",
            "brier",
        ]
    ].rename(index=labels)
)
fig, axes = plt.subplots(1, 4, figsize=(14, 4), layout="constrained")
for ax, (name, score) in zip(axes, scores.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_validation,
        score >= thresholds[name],
        labels=[0, 1],
        display_labels=["No rejection", "Rejection"],
        colorbar=False,
        cmap="Blues",
        ax=ax,
        values_format="d",
    )
    ax.set(title=labels[name], xlabel="Model flag", ylabel="Recorded diagnosis")
    ax.tick_params(labelsize=9)
session.figure("confusion_matrices", fig)
plt.show()
session.table("validation_metrics", metric_table)
display(
    Markdown(
        f"**Observed:** CatBoost missed {int(metric_table.loc[selected, 'fn'])} of {sum(y_validation)} rejection cases "
        f"and flagged {int(metric_table.loc[selected, 'fp'])} of {sum(y_validation == 0)} no-rejection specimens. "
        f"Logistic regression missed {int(metric_table.loc['logistic_all', 'fn'])} with "
        f"{int(metric_table.loc['logistic_all', 'fp'])} false flags. "
        "CatBoost's screening sensitivity target was not maintained in technical validation."
    )
)

## 4. Compare score ranking, uncertainty, and reliability

ROC and precision–recall curves describe score ranking across thresholds. The
evaluation prevalence is the precision–recall reference. Reliability compares
mean scores with observed rejection fractions in fixed bins; this plot does
not fit a calibration correction.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), layout="constrained")
reliability_rows = []
for (name, score), color in zip(scores.items(), COLORS):
    fpr, tpr, _ = roc_curve(y_validation, score)
    precision, recall, _ = precision_recall_curve(y_validation, score)
    axes[0].plot(fpr, tpr, color=color, label=labels[name])
    if name == "training_prevalence":
        axes[1].scatter([1], [y_validation.mean()], color=color, label=labels[name])
    else:
        axes[1].plot(recall, precision, color=color, label=labels[name])
    rows = reliability(y_validation, score, name)
    reliability_rows.extend(rows)
    bins = pd.DataFrame(rows).query("n > 0")
    axes[2].plot(
        bins.mean_score, bins.observed_fraction, "o-", color=color, label=labels[name], markersize=4
    )
axes[0].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[1].axhline(
    y_validation.mean(), color="gray", linestyle="--", label=f"Prevalence {y_validation.mean():.1%}"
)
axes[2].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[0].set(title="ROC", xlabel="False-positive rate", ylabel="Sensitivity")
axes[1].set(title="Precision–recall", xlabel="Recall / sensitivity", ylabel="Precision")
axes[2].set(
    title="Score reliability (10 fixed bins)",
    xlabel="Mean model score",
    ylabel="Observed rejection fraction",
)
for ax in axes:
    ax.set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.02))
    ax.legend(fontsize=8)
session.figure("ranking_and_reliability", fig)
plt.show()
session.table("score_reliability_bins", pd.DataFrame(reliability_rows))

In [ ]:
_, intervals, differences = paired_bootstrap(
    y_validation,
    {name: scores[name] for name in [selected, "logistic_all"]},
    thresholds,
    n_bootstrap=2000,
    seed=20260915,
)
display(differences.loc[differences.metric.isin(["sensitivity", "specificity", "fn", "fp"])])
session.table("paired_model_differences", differences)
session.table("model_intervals", intervals)

The paired bootstrap resamples the same **specimen positions** for both models
in each of 2,000 replicates. It describes uncertainty conditional on these
fitted models and this cohort, without verifying patient independence or
incorporating model-training uncertainty. The existing analysis found that the
sensitivity-difference interval includes zero. Continue to call the output a
**model score**; the reliability plot alone does not establish calibrated probabilities.

The [discovery stability follow-up](../results/followup/20260917_stability/REPORT.md)
also found that model preference varied across development splits.

## 5. Which assay targets have the largest saved CatBoost importance?

CatBoost's [PredictionValuesChange](https://catboost.ai/docs/en/concepts/fstr)
is a nonnegative importance measure based on changes between tree leaves.
Its values sum to 100. It ranks overall use of targets but supplies neither a
signed effect nor a percentage of biological causation.

In [ ]:
native_importance = (
    pd.read_csv(session.artifact("any_rejection_gene_importance.csv")).set_index("gene").importance
)
recomputed_importance = pd.Series(
    models[selected].get_feature_importance(type="PredictionValuesChange"), index=X.columns
)
np.testing.assert_allclose(
    recomputed_importance.loc[native_importance.index], native_importance, atol=1e-10, rtol=0
)
top_native = native_importance.nlargest(20).sort_values()
fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")
ax.barh(
    top_native.index,
    top_native,
    color=[COLORS[1] if t in BK_TARGETS else COLORS[0] for t in top_native.index],
)
ax.set(
    xlabel="PredictionValuesChange (all 758 targets sum to 100)",
    title="CatBoost: 20 highest-ranked assay targets",
)
ax.text(0.99, 0.04, "Orange: BK viral targets", transform=ax.transAxes, ha="right", color=COLORS[1])
session.figure("catboost_native_importance", fig)
plt.show()
display(native_importance.head(10).rename("importance").to_frame())
session.table("catboost_native_importance", native_importance.rename("importance").to_frame())

## 6. Which targets contribute most to screening scores?

[SHAP values](https://catboost.ai/docs/en/concepts/shap-values) assign a contribution
to each target for each specimen. We use CatBoost's built-in calculation with
its tree-based reference, without supplying a new background cohort. For this
binary model, contributions plus the expected value reconstruct the raw **log-odds**
score. Applying the logistic function reconstructs the 0–1 model score.

Rank targets by mean absolute contribution across all 263 screening specimens.
This is a descriptive ranking, not a new feature-selection step. A target's
importance can differ between this ranking and the saved tree importance.

In [ ]:
shap_screen = models[selected].get_feature_importance(
    Pool(X.loc[screen_ids]), type="ShapValues", thread_count=3
)
raw_screen = models[selected].predict(X.loc[screen_ids], prediction_type="RawFormulaVal")
np.testing.assert_allclose(shap_screen.sum(axis=1), raw_screen, rtol=0, atol=1e-10)
np.testing.assert_allclose(
    expit(shap_screen.sum(axis=1)),
    predict_scores(models[selected], X.loc[screen_ids]),
    rtol=0,
    atol=1e-10,
)
contributions = pd.DataFrame(shap_screen[:, :-1], index=screen_ids, columns=X.columns)
mean_abs_shap = contributions.abs().mean().sort_values(ascending=False)
ranks = pd.DataFrame({"mean_absolute_SHAP": mean_abs_shap, "native_importance": native_importance})
ranks["SHAP_rank"] = ranks.mean_absolute_SHAP.rank(ascending=False, method="min").astype(int)
ranks["native_rank"] = ranks.native_importance.rank(ascending=False, method="min").astype(int)
display(ranks.sort_values("SHAP_rank").head(15))
session.table("catboost_screening_contributions", ranks.sort_values("SHAP_rank"))
print("SHAP contributions reconstruct raw scores and 0–1 scores within 1e-10.")

In [ ]:
top_shap = mean_abs_shap.head(15).index[::-1]
rng = np.random.default_rng(20260927)
fig, axes = plt.subplots(1, 2, figsize=(14, 7), layout="constrained", sharey=True)
axes[0].barh(np.arange(len(top_shap)), mean_abs_shap.loc[top_shap], color=COLORS[0])
axes[0].set_yticks(np.arange(len(top_shap)), top_shap)
axes[0].set(
    xlabel="Mean absolute contribution (log-odds)", title="Overall contribution on screening"
)
for position, target in enumerate(top_shap):
    percentile = X.loc[screen_ids, target].rank(pct=True).to_numpy()
    dots = axes[1].scatter(
        contributions[target],
        position + rng.uniform(-0.28, 0.28, len(screen_ids)),
        c=percentile,
        cmap="coolwarm",
        vmin=0,
        vmax=1,
        s=12,
        alpha=0.65,
    )
axes[1].axvline(0, color="gray", linewidth=1)
axes[1].set(xlabel="Contribution to rejection log-odds", title="Direction varies across specimens")
fig.colorbar(dots, ax=axes[1], label="Within-target expression percentile", shrink=0.7)
session.figure("shap_global_and_direction", fig)
plt.show()
display(
    Markdown(
        f"**Observed:** the largest mean absolute screening contribution is from **{mean_abs_shap.index[0]}**. "
        "Points to the right increase this model's rejection score relative to its expected value; "
        "points to the left decrease it. Redder points have higher expression within that target. "
        "These are model explanations, not causal effects of changing RNA."
    )
)

## 7. What does logistic regression emphasize?

Each coefficient below is the fitted change in log-odds per one **training standard
deviation** increase in a normalized target, holding the other inputs fixed.
Positive coefficients increase the model score; negative coefficients decrease it.
L2 regularization and correlated RNA measurements affect their magnitudes and signs.
These coefficients and CatBoost importances use different scales.

In [ ]:
logistic = models["logistic_all"]
coefficients = pd.Series(
    logistic.named_steps["logisticregression"].coef_[0], index=X.columns, name="coefficient"
)
top_coefficients = coefficients.loc[coefficients.abs().nlargest(20).index].sort_values()
fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")
ax.barh(
    top_coefficients.index,
    top_coefficients,
    color=[COLORS[1] if v > 0 else COLORS[0] for v in top_coefficients],
)
ax.axvline(0, color="gray", linewidth=1)
ax.set(
    xlabel="Log-odds coefficient per training SD",
    title="Logistic regression: 20 largest absolute coefficients",
)
session.figure("logistic_coefficients", fig)
plt.show()
session.table("logistic_standardized_coefficients", coefficients.to_frame())

## 8. Inspect distributions of the leading contributing targets

These are the four leading targets by screening mean absolute SHAP value.
The boxes show discovery specimens grouped by recorded diagnosis. A difference
between groups does not establish a unique or causal contribution, especially
when targets are correlated. Viral measurements remain labeled as assay targets.

In [ ]:
leading_targets = mean_abs_shap.head(4).index
discovery_ids = split.index[split.split.ne("author_validation")]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
for ax, target in zip(axes.flat, leading_targets):
    ax.boxplot(
        [
            X.loc[discovery_ids[diagnosis.loc[discovery_ids].eq(label)], target]
            for label in DIAGNOSIS_ORDER
        ],
        tick_labels=SHORT_DIAGNOSES,
        flierprops={"markersize": 2, "alpha": 0.3},
    )
    ax.set(
        title=target + (" (viral target)" if target in BK_TARGETS else ""),
        ylabel="Normalized assay value",
    )
    ax.tick_params(axis="x", rotation=15, labelsize=9)
session.figure("leading_targets_by_diagnosis", fig)
plt.show()

## 9. Explain individual correct and incorrect predictions

Select the **first specimen in sorted accession order** in each of the four
outcome groups. This deterministic choice avoids selecting unusually persuasive
explanations, but four cases are not a representative sample. Full specimen
records remain in memory; displayed examples use outcome labels.

Each plot shows the ten largest absolute contributions plus the sum of all
remaining targets. Positive bars increase log-odds; negative bars decrease them.
The expected value is reported separately. A false flag or miss is an error
against the recorded diagnosis, and SHAP does not establish why that diagnosis
and score disagree.

In [ ]:
flags = scores[selected] >= thresholds[selected]
outcomes = pd.Series(
    np.select(
        [(y_validation == 1) & flags, (y_validation == 1) & ~flags, (y_validation == 0) & flags],
        ["Correct rejection", "Missed rejection", "False rejection flag"],
        default="Correct no rejection",
    ),
    index=validation_ids,
)
outcome_order = [
    "Correct no rejection",
    "Correct rejection",
    "False rejection flag",
    "Missed rejection",
]
case_ids = [sorted(outcomes.index[outcomes.eq(group)])[0] for group in outcome_order]
shap_cases = models[selected].get_feature_importance(
    Pool(X.loc[case_ids]), type="ShapValues", thread_count=3
)
case_scores = predict_scores(models[selected], X.loc[case_ids])
np.testing.assert_allclose(expit(shap_cases.sum(axis=1)), case_scores, rtol=0, atol=1e-10)
fig, axes = plt.subplots(2, 2, figsize=(14, 11), layout="constrained")
for index, (ax, group, specimen) in enumerate(zip(axes.flat, outcome_order, case_ids)):
    contribution = pd.Series(shap_cases[index, :-1], index=X.columns)
    largest = contribution.abs().nlargest(10).index
    shown = contribution.loc[largest].copy()
    shown.loc["All other targets"] = contribution.drop(largest).sum()
    shown = shown.sort_values()
    ax.barh(shown.index, shown, color=[COLORS[1] if v > 0 else COLORS[0] for v in shown])
    ax.axvline(0, color="gray", linewidth=1)
    ax.set(
        xlabel="Contribution to rejection log-odds",
        title=f"{group}\nScore {case_scores[index]:.3f}; threshold {thresholds[selected]:.3f}\nExpected log-odds {shap_cases[index, -1]:.3f}",
    )
    ax.tick_params(axis="y", labelsize=9)
session.figure("individual_prediction_contributions", fig)
plt.show()
subtype_errors = pd.DataFrame(
    {
        "diagnosis": diagnosis.loc[validation_ids],
        "missed": (y_validation == 1) & ~flags,
        "false_flag": (y_validation == 0) & flags,
    }
)
subtype_errors = subtype_errors.groupby("diagnosis").agg(
    specimens=("missed", "size"), missed=("missed", "sum"), false_flags=("false_flag", "sum")
)
display(subtype_errors)
session.table("errors_by_diagnosis", subtype_errors)

## 10. Conclusions

The preserved CatBoost model made fewer missed-rejection errors than logistic
regression in this evaluation, with the same number of false flags. The difference
is uncertain, and development-split stability does not establish a dependable
preference.

The contribution plots make the role of particular assay targets visible.
Correlated targets, viral signals, and the mix of specimens limit biological
interpretation. Neither feature importance nor agreement with recorded diagnoses
establishes that the score improves treatment decisions.

Further evidence: [primary analysis](../results/analysis/20260915_baseline/REPORT.md),
[viral review](../results/analysis/20260915_viral/REPORT.md),
[shared scoring code](../src/kidney_biopsy/prediction.py),
and [research context](../docs/RESEARCH_CONTEXT.md).

In [ ]:
session.finish(
    {
        "model_version": predictor.model_version,
        "validation_rows": len(validation_ids),
        "catboost_missed": int(metric_table.loc[selected, "fn"]),
        "catboost_false_flags": int(metric_table.loc[selected, "fp"]),
        "logistic_missed": int(metric_table.loc["logistic_all", "fn"]),
        "highest_native_importance": native_importance.idxmax(),
        "highest_screening_mean_abs_shap": mean_abs_shap.index[0],
        "score_and_shap_checks_passed": True,
    },
    {
        "shap_population": "All 263 discovery-screen specimens",
        "shap_method": "CatBoost ShapValues, no reference_data, raw log-odds",
        "case_rule": "First sorted accession in each of four validation outcome groups",
        "bootstrap_replicates": 2000,
        "bootstrap_seed": 20260915,
        "score_tolerance": 1e-10,
        "thresholds": thresholds,
    },
)